# SciQ Facts Index Builder

Build `Indexes/science_nature_facts_sciq_v1` from `Datasets/science_nature_facts_sciq_v1`.

This index is the first lightweight `facts` index for the Science/Nature RAG. It uses one SciQ `support` passage as one FAISS document and keeps the agreed metadata:

- `doc_id`
- `source_dataset`
- `source_split`
- `source_type`
- `license`
- `subject`
- `topic`
- `taxonomy_source`
- `taxonomy_confidence`


In [ ]:
# @title Mount Google Drive

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/NLP")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/NLP")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Non trovo la cartella progetto. "
        "Controlla se il path è /content/drive/MyDrive/NLP oppure modifica PROJECT_ROOT manualmente."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

## 0. Setup

If running in Colab and dependencies are missing, uncomment the install cell. The embedding endpoint must be running before the build cell.

In [ ]:
!pip -q install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu datasets pandas tqdm

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil

import pandas as pd
from datasets import DatasetDict, load_from_disk
from tqdm.auto import tqdm

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Example for Colab/Drive:
# PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/NLP"
PROJECT_ROOT_OVERRIDE = None

PROJECT_MARKERS = [Path("requirements.txt"), Path("millionaire_client")]
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path("/content/drive/MyDrive/NLP"),
    Path("/content/drive/MyDrive/Colab Notebooks/NLP"),
    Path("/gdrive/MyDrive/NLP"),
    Path("/gdrive/MyDrive/Colab Notebooks/NLP"),
]

def looks_like_project_root(path):
    return any((path / marker).exists() for marker in PROJECT_MARKERS)

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser()
else:
    PROJECT_ROOT = next((candidate for candidate in PROJECT_ROOT_CANDIDATES if looks_like_project_root(candidate)), Path.cwd())

DATASETS_DIR = PROJECT_ROOT / "Datasets"
INDEXES_DIR = PROJECT_ROOT / "Indexes"
LOGS_DIR = PROJECT_ROOT / "logs"
INDEXES_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DATASET_DIR = DATASETS_DIR / "science_nature_facts_sciq_v1"
OUTPUT_INDEX_DIR = INDEXES_DIR / "science_nature_facts_sciq_v1"
INDEX_REPORT_JSON = LOGS_DIR / "science_nature_facts_sciq_v1_index_report.json"

# Same embedding model used in the project settings/Ancient pipeline.
EMBEDDING_MODEL = "hf.co/unsloth/embeddinggemma-300m-GGUF:BF16"
EMBEDDING_BASE_URL = "http://localhost:11434/v1/"
EMBEDDING_API_KEY = "ollama"

BATCH_SIZE = 64
OVERWRITE_INDEX = False
RUN_SMOKE_TESTS = True

print("project root:", PROJECT_ROOT)
print("input dataset:", INPUT_DATASET_DIR)
print("output index:", OUTPUT_INDEX_DIR)
print("embedding model:", EMBEDDING_MODEL)

## 1. Load Dataset

The dataset must already exist. If the builder was run in Colab, run this notebook in the same Drive-backed project folder or copy the dataset directory into this project first.

In [ ]:
if not INPUT_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_DATASET_DIR}\n"
        "Run sciq_builder.ipynb first in this project root, or set PROJECT_ROOT_OVERRIDE to the Drive folder where the dataset was saved."
    )

loaded = load_from_disk(str(INPUT_DATASET_DIR))
ds = loaded["train"] if isinstance(loaded, DatasetDict) else loaded
df = ds.to_pandas().fillna("")

required_cols = [
    "doc_id",
    "text",
    "source_dataset",
    "source_split",
    "source_type",
    "license",
    "subject",
    "topic",
    "taxonomy_source",
    "taxonomy_confidence",
]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy().reset_index(drop=True)

print("rows to index:", len(df))
display(df[required_cols].head())
display(df["subject"].value_counts().rename_axis("subject").reset_index(name="rows"))

## 2. Build Documents

For this first facts index, we do not chunk SciQ supports. One support passage becomes one vector document.

In [ ]:
METADATA_COLS = [
    "doc_id",
    "source_dataset",
    "source_split",
    "source_type",
    "license",
    "subject",
    "topic",
    "taxonomy_source",
    "taxonomy_confidence",
]

# Optional non-leaky technical metadata for debugging/dedup checks.
OPTIONAL_METADATA_COLS = ["support_hash", "taxonomy_hits"]

def clean_metadata_value(value, col):
    if col == "taxonomy_confidence":
        try:
            return float(value)
        except Exception:
            return 0.0
    if value is None:
        return ""
    return str(value)

docs = []
metadata_cols = [col for col in METADATA_COLS + OPTIONAL_METADATA_COLS if col in df.columns]

for _, row in df.iterrows():
    meta = {col: clean_metadata_value(row.get(col, ""), col) for col in metadata_cols}
    docs.append(Document(page_content=str(row["text"]), metadata=meta))

print("documents:", len(docs))
print("metadata cols:", metadata_cols)
print("sample doc:")
print(docs[0].page_content[:500])
print(json.dumps(docs[0].metadata, indent=2, ensure_ascii=False))

In [ ]:
# Install Ollama, solo se nel runtime non c'è già
!apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# Avvia Ollama in background
import subprocess, time

subprocess.run(["killall", "ollama"], check=False)

with open("/tmp/ollama_sciq_index.log", "ab") as log_file:
    subprocess.Popen(
        ["bash", "-lc", "OLLAMA_CONTEXT_LENGTH=8192 ollama serve"],
        stdout=log_file,
        stderr=log_file,
    )

time.sleep(5)

In [ ]:
# Scarica il modello embedding usato dal notebook
EMBEDDING_MODEL = "hf.co/unsloth/embeddinggemma-300m-GGUF:BF16"

!ollama pull {EMBEDDING_MODEL}
!ollama list

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama",
)

resp = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="friction can oppose motion",
)

print("embedding dimension:", len(resp.data[0].embedding))

## 3. Build FAISS Index

This cell calls the local Ollama/OpenAI-compatible embedding endpoint. Start the embedding server first if needed.

In [ ]:
if OUTPUT_INDEX_DIR.exists():
    if not OVERWRITE_INDEX:
        raise FileExistsError(f"{OUTPUT_INDEX_DIR} exists. Set OVERWRITE_INDEX=True to replace it.")
    shutil.rmtree(OUTPUT_INDEX_DIR)

if not docs:
    raise RuntimeError("No documents to index.")

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=EMBEDDING_BASE_URL,
    api_key=EMBEDDING_API_KEY,
    check_embedding_ctx_length=False,
)

vectorstore = None
for start in tqdm(range(0, len(docs), BATCH_SIZE), desc="embedding SciQ facts"):
    batch = docs[start:start + BATCH_SIZE]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)

vectorstore.save_local(str(OUTPUT_INDEX_DIR))

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "full_sciq_facts_index",
    "input_dataset_dir": str(INPUT_DATASET_DIR),
    "output_index_dir": str(OUTPUT_INDEX_DIR),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_base_url": EMBEDDING_BASE_URL,
    "batch_size": BATCH_SIZE,
    "rows_indexed": len(df),
    "documents_indexed": len(docs),
    "metadata_cols": metadata_cols,
    "subject_counts": df["subject"].value_counts().to_dict(),
    "index_files": sorted(p.name for p in OUTPUT_INDEX_DIR.iterdir()),
}
INDEX_REPORT_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

print("saved index:", OUTPUT_INDEX_DIR)
print("saved report:", INDEX_REPORT_JSON)
print(json.dumps(report, indent=2, ensure_ascii=False))

## 4. Smoke Test

A few retrieval checks for the kind of Science/Nature questions we expect.

In [ ]:
if RUN_SMOKE_TESTS:
    if not OUTPUT_INDEX_DIR.exists():
        raise FileNotFoundError(f"Index not found: {OUTPUT_INDEX_DIR}")

    test_store = FAISS.load_local(str(OUTPUT_INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
    queries = [
        "What force can stop the motion of a rolling ball? Options: gravity | friction | sunlight | sound",
        "A sodium atom transfers an electron to chlorine. What happens to the sodium atom? Options: neutral | positive ion | negative ion | isotope",
        "Which would most likely cause a decrease in woodpeckers in an ecosystem? Options: more food | fewer trees | more nesting sites | less competition",
    ]

    rows = []
    for query in queries:
        results = test_store.similarity_search_with_score(query, k=5)
        for rank, (doc, score) in enumerate(results, start=1):
            rows.append({
                "query": query,
                "rank": rank,
                "score": float(score),
                "subject": doc.metadata.get("subject", ""),
                "topic": doc.metadata.get("topic", ""),
                "source_split": doc.metadata.get("source_split", ""),
                "doc_id": doc.metadata.get("doc_id", ""),
                "text_preview": doc.page_content[:300],
            })

    smoke_df = pd.DataFrame(rows)
    display(smoke_df)
else:
    print("Skipping smoke tests because RUN_SMOKE_TESTS=False")

## Next Runtime Config

Once this index exists, the science agent can start with:

```python
"science": {
    "prompt": "science",
    "tools": {"wikipedia_search": False},
    "rag": {
        "enabled": True,
        "index_directory": "Indexes/science_nature_facts_sciq_v1",
        "dataset": "science_nature_facts_sciq_v1",
        "topk": 4,
    },
}
```

Later, when the Wikipedia index exists too, replace this with a hybrid retriever that queries both indices.